# Ablation Study

Systematically measures the contribution of each training component.

| Row | Pseudo-label | GRPO | QC post-proc | Description |
|---|---|---|---|---|
| A | — | — | — | Qwen3-0.6B base (zero-shot) |
| B | ✗ | ✗ | ✗ | SFT on raw teacher labels |
| C | ✓ | ✗ | ✗ | SFT on pseudo-labeled data |
| D | ✓ | ✗ | ✓ | SFT + QC post-processing |
| E | ✓ | ✓ | ✗ | SFT + GRPO |
| F | ✓ | ✓ | ✓ | SFT + GRPO + QC (full pipeline) |

Each row reads pre-saved predictions from prior runs.
Run the corresponding notebooks/scripts first, then save predictions
as `ablation/<row>_predictions.csv` with columns `[note, LLM-XML, GT-XML]`.

In [ ]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = "/kaggle/input/datasets/sinadaneshgar/edgedeid"  # Kaggle dataset fallback
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

## 0 — Setup

In [ ]:
import os, sys

import pandas as pd
from edgedeid.eval import compute_metrics, print_report

ABLATION_DIR = "ablation"   # directory holding per-row prediction CSVs
os.makedirs(ABLATION_DIR, exist_ok=True)

# Ordered ablation rows — (key, label, csv_filename)
ABLATION_ROWS = [
    ("A", "Base (0-shot)",             "A_base_zeroshot.csv"),
    ("B", "SFT (raw labels)",           "B_sft_raw.csv"),
    ("C", "SFT (pseudo-label)",         "C_sft_pseudo.csv"),
    ("D", "SFT (pseudo) + QC",          "D_sft_pseudo_qc.csv"),
    ("E", "SFT + GRPO",                 "E_sft_grpo.csv"),
    ("F", "SFT + GRPO + QC (full)",     "F_sft_grpo_qc.csv"),
]

## 1 — Load Predictions

Each CSV must have columns: `note`, `LLM-XML`, `GT-XML`.

**How to generate each file:**

- **A** (base zero-shot): serve `Qwen/Qwen3-0.6B-Base` with vLLM, run `synth_eval.ipynb` with that model.
- **B** (SFT raw): serve `Qwen3-0.6B-Base_SFT` trained on raw labels (run `combine.ipynb` with `PSEUDO_LABEL=False` first).
- **C** (SFT pseudo): use existing `synth_eval/synth_eval_predictions.csv` — this is the main SFT checkpoint.
- **D** (SFT + QC): apply QC post-processing script to row C predictions. *TODO — ...*.
- **E** (SFT + GRPO): serve GRPO-trained model, run `synth_eval.ipynb`.
- **F** (full): apply QC post-processing to row E predictions. *TODO — ...*.

In [ ]:
ablation_data = {}   # key -> (predictions, ground_truths, notes)

for key, label, fname in ABLATION_ROWS:
    path = os.path.join(ABLATION_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        ablation_data[key] = (
            df["LLM-XML"].tolist(),
            df["GT-XML"].tolist(),
            df["note"].tolist() if "note" in df.columns else None,
        )
        print(f"  [{key}] {label}: {len(df)} samples")
    else:
        print(f"  [{key}] {label}: *** NOT FOUND — skipping ({path}) ***")

print(f"\n{len(ablation_data)} / {len(ABLATION_ROWS)} rows loaded.")

## 2 — Compute Metrics for Each Row

In [ ]:
results = {}   # key -> EvalResult

for key, label, _ in ABLATION_ROWS:
    if key not in ablation_data:
        continue
    preds, gts, notes = ablation_data[key]
    r = compute_metrics(preds, gts, source_notes=notes)
    results[key] = (label, r)
    print_report(r, title=f"[{key}] {label}")
    print()

## 3 — Ablation Table

In [ ]:
rows = []
for key, label, _ in ABLATION_ROWS:
    if key not in results:
        rows.append({"Row": key, "Config": label,
                     "Micro-P": None, "Micro-R": None, "Micro-F1": None,
                     "Macro-F1": None, "XML-Parse": None, "Hallucination": None})
        continue
    _, r = results[key]
    rows.append({
        "Row":           key,
        "Config":        label,
        "Micro-P":       round(r.micro_precision, 4),
        "Micro-R":       round(r.micro_recall, 4),
        "Micro-F1":      round(r.micro_f1, 4),
        "Macro-F1":      round(r.macro_f1, 4),
        "XML-Parse":     round(r.xml_parse_rate, 4),
        "Hallucination": round(r.hallucination_rate, 4),
    })

ablation_df = pd.DataFrame(rows).set_index("Row")
ablation_df

## 4 — Δ vs Previous Row (Incremental Gain)

In [ ]:
numeric_cols = ["Micro-F1", "Macro-F1", "XML-Parse", "Hallucination"]
delta_df = ablation_df[numeric_cols].diff().round(4)
delta_df.columns = [f"Δ {c}" for c in delta_df.columns]
pd.concat([ablation_df, delta_df], axis=1)

## 5 — Export

In [ ]:
import os
os.makedirs("results", exist_ok=True)
ablation_df.to_csv("results/ablation_results.csv")
print("Saved results/ablation_results.csv")